# 第五节课实验：守恒量约束 PINN 求解 1D 线性平流方程

本 Notebook 目标：

1. 从热传导方程切换到一个更适合讲“守恒量”的 PDE；
2. 先搭建普通 PINN 作为基线；
3. 在标准 PINN 的基础上加入守恒量约束项；
4. 比较普通 PINN 与 Conservation-PINN 在误差和守恒保持方面的表现。

## 1. 问题定义

我们考虑 1D 线性平流方程：

$$u_t + c u_x = 0, \quad x\in[0,1],\ t\in[0,1]$$

其中 $c$ 是常数传播速度。

为了让问题更适合讲“守恒量”，我们采用周期边界条件：

$$u(0,t)=u(1,t)$$

必要时还可以加入导数的周期一致性：

$$u_x(0,t)=u_x(1,t)$$

本节课选取一个总量不为零的初值，这样更容易观察守恒量是否漂移：

$$u(x,0)=1+0.5\sin(2\pi x)$$

对于这个初值，总质量定义为：

$$M(t)=\int_0^1 u(x,t)\,dx$$

理论上应满足：

$$M(t)=M(0)$$

解析解为：

$$u(x,t)=1+0.5\sin\big(2\pi(x-ct)\big)$$

## 2. 方法说明

### 2.1 普通 PINN

用神经网络表示未知解：

$$u_\theta(x,t)$$

PDE 残差定义为：

$$f_\theta(x,t)=u_t + c u_x$$

普通 PINN 的总损失写成：

$$\mathcal{L}_{base}=\lambda_{ic}\mathcal{L}_{ic}+\lambda_{bc}\mathcal{L}_{bc}+\lambda_f\mathcal{L}_f$$

其中：

$$\mathcal{L}_{ic}=\frac{1}{N_{ic}}\sum_{i=1}^{N_{ic}}\left|u_\theta(x_i,0)-u_0(x_i)\right|^2$$

$$\mathcal{L}_{bc}=\frac{1}{N_{bc}}\sum_{i=1}^{N_{bc}}\left|u_\theta(0,t_i)-u_\theta(1,t_i)\right|^2+\frac{1}{N_{bc}}\sum_{i=1}^{N_{bc}}\left|u_x(0,t_i)-u_x(1,t_i)\right|^2$$

$$\mathcal{L}_f=\frac{1}{N_f}\sum_{j=1}^{N_f}\left|u_t(x_j,t_j)+c u_x(x_j,t_j)\right|^2$$

### 2.2 守恒量约束 PINN

我们希望模型不仅满足点上的 PDE 残差，还能尽量保持整体质量守恒：

$$M_\theta(t)=\int_0^1 u_\theta(x,t)\,dx$$

引入守恒量约束项：

$$\mathcal{L}_{cons}=\frac{1}{N_t}\sum_{n=1}^{N_t}\left|M_\theta(t_n)-M_0\right|^2$$

其中：

$$M_0=\int_0^1 u(x,0)\,dx$$

于是改进模型的总损失为：

$$\mathcal{L}=\lambda_{ic}\mathcal{L}_{ic}+\lambda_{bc}\mathcal{L}_{bc}+\lambda_f\mathcal{L}_f+\lambda_{cons}\mathcal{L}_{cons}$$

实际实现时，我们用数值积分近似：

$$M_\theta(t_n)\approx \frac{1}{N_x}\sum_{k=1}^{N_x}u_\theta(x_k,t_n)$$

这里 $x_k$ 取 $[0,1]$ 上的一组均匀点即可。

## 3. 导入库与基础设置

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Device:', device)

## 4. 参数设置

这一版先给出一个适合课堂演示和后续改实验的默认配置。

In [ ]:
c = 1.0

# 训练点数量
N_ic = 128
N_bc = 128
N_f = 2000

# 守恒约束使用的时间采样点和积分网格点
N_cons_t = 50
N_cons_x = 200

# 网络与优化参数
hidden_dim = 64
num_hidden = 4
lr = 1e-3
epochs = 8000

# 损失权重
lambda_ic = 1.0
lambda_bc = 1.0
lambda_f = 1.0
lambda_cons = 1.0

print('c =', c)
print('N_ic =', N_ic)
print('N_bc =', N_bc)
print('N_f =', N_f)
print('N_cons_t =', N_cons_t)
print('N_cons_x =', N_cons_x)
print('hidden_dim =', hidden_dim)
print('num_hidden =', num_hidden)
print('lr =', lr)
print('epochs =', epochs)
print('lambda_cons =', lambda_cons)

## 5. 解析解与初值函数

In [ ]:
def initial_condition(x):
    return 1.0 + 0.5 * np.sin(2.0 * np.pi * x)


def exact_solution(x, t, c):
    return 1.0 + 0.5 * np.sin(2.0 * np.pi * (x - c * t))


# 理论总质量 M0
M0_exact = 1.0
print('Exact conserved mass M0 =', M0_exact)

## 6. 构造训练点

- 初值点：$(x,0)$
- 周期边界点：$(0,t)$ 和 $(1,t)$
- 内部 collocation 点：$(x_f,t_f)$
- 守恒量约束点：若干个时间切片 $t_n$ 与积分网格 $x_k$

In [ ]:
# IC points
x_ic = np.random.rand(N_ic, 1)
t_ic = np.zeros((N_ic, 1))
u_ic = initial_condition(x_ic)

# Periodic BC points
t_bc = np.random.rand(N_bc, 1)
x_bc0 = np.zeros((N_bc, 1))
x_bc1 = np.ones((N_bc, 1))

# Collocation points
x_f = np.random.rand(N_f, 1)
t_f = np.random.rand(N_f, 1)

# Conservation points: sample times + uniform x-grid
cons_t = np.linspace(0.0, 1.0, N_cons_t).reshape(-1, 1)
cons_x = np.linspace(0.0, 1.0, N_cons_x).reshape(-1, 1)


def to_tensor(a, requires_grad=False):
    return torch.tensor(a, dtype=torch.float32, device=device, requires_grad=requires_grad)


x_ic_t = to_tensor(x_ic)
t_ic_t = to_tensor(t_ic)
u_ic_t = to_tensor(u_ic)

t_bc_t = to_tensor(t_bc)
x_bc0_t = to_tensor(x_bc0, requires_grad=True)
x_bc1_t = to_tensor(x_bc1, requires_grad=True)

x_f_t = to_tensor(x_f, requires_grad=True)
t_f_t = to_tensor(t_f, requires_grad=True)

cons_t_t = to_tensor(cons_t)
cons_x_t = to_tensor(cons_x)

print('IC points:', x_ic_t.shape[0])
print('BC points:', t_bc_t.shape[0])
print('F points :', x_f_t.shape[0])
print('Conservation time slices:', cons_t_t.shape[0])
print('Conservation x-grid size:', cons_x_t.shape[0])

## 7. 定义 PINN 网络

这一版先用最常见的 `MLP + tanh`。如果后面想扩展到 Res-PINN，可以直接替换这里的网络结构。

In [ ]:
class PINN(nn.Module):
    def __init__(self, in_dim=2, out_dim=1, hidden_dim=64, num_hidden=4):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.Tanh()]
        for _ in range(num_hidden - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers += [nn.Linear(hidden_dim, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)


baseline_model = PINN(hidden_dim=hidden_dim, num_hidden=num_hidden).to(device)
cons_model = PINN(hidden_dim=hidden_dim, num_hidden=num_hidden).to(device)

print('Baseline PINN:')
print(baseline_model)

## 8. 自动微分与 PDE 残差

In [ ]:
def gradients(y, x):
    return torch.autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True
    )[0]


def pde_residual(model, x, t, c):
    u = model(x, t)
    u_t = gradients(u, t)
    u_x = gradients(u, x)
    return u_t + c * u_x

## 9. 边界损失与守恒量损失

周期边界这里同时约束函数值和一阶导数。

In [ ]:
def periodic_bc_loss(model, t_bc):
    x0 = torch.zeros_like(t_bc, requires_grad=True)
    x1 = torch.ones_like(t_bc, requires_grad=True)

    u0 = model(x0, t_bc)
    u1 = model(x1, t_bc)
    ux0 = gradients(u0, x0)
    ux1 = gradients(u1, x1)

    loss_u = torch.mean((u0 - u1) ** 2)
    loss_ux = torch.mean((ux0 - ux1) ** 2)
    return loss_u + loss_ux


def conservation_mass(model, cons_x_t, t_value):
    t_column = torch.full_like(cons_x_t, float(t_value))
    u_vals = model(cons_x_t, t_column)
    return torch.mean(u_vals)


def conservation_loss(model, cons_x_t, cons_t_t, M0):
    mass_errors = []
    masses = []

    for i in range(cons_t_t.shape[0]):
        t_value = cons_t_t[i, 0].item()
        mass_t = conservation_mass(model, cons_x_t, t_value)
        masses.append(mass_t)
        mass_errors.append((mass_t - M0) ** 2)

    loss_cons = torch.mean(torch.stack(mass_errors))
    masses = torch.stack(masses)
    return loss_cons, masses

## 10. 统一训练函数

这里做两种模式：

- `use_conservation=False`：普通 PINN
- `use_conservation=True`：加入守恒量约束

In [ ]:
def train_model(model, model_name, use_conservation=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {
        'total': [],
        'ic': [],
        'bc': [],
        'f': [],
        'cons': [],
    }

    start = time.time()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()

        # IC loss
        u_ic_pred = model(x_ic_t, t_ic_t)
        loss_ic = torch.mean((u_ic_pred - u_ic_t) ** 2)

        # BC loss
        loss_bc = periodic_bc_loss(model, t_bc_t)

        # PDE loss
        f = pde_residual(model, x_f_t, t_f_t, c)
        loss_f = torch.mean(f ** 2)

        # Conservation loss
        if use_conservation:
            loss_cons, _ = conservation_loss(model, cons_x_t, cons_t_t, M0_exact)
        else:
            loss_cons = torch.tensor(0.0, dtype=torch.float32, device=device)

        loss = (
            lambda_ic * loss_ic
            + lambda_bc * loss_bc
            + lambda_f * loss_f
            + lambda_cons * loss_cons
        )

        loss.backward()
        optimizer.step()

        history['total'].append(loss.item())
        history['ic'].append(loss_ic.item())
        history['bc'].append(loss_bc.item())
        history['f'].append(loss_f.item())
        history['cons'].append(loss_cons.item())

        if epoch % 200 == 0 or epoch == 1:
            print(
                f'[{model_name}] Epoch {epoch:4d} | '
                f'Total {loss.item():.4e} | '
                f'IC {loss_ic.item():.4e} | '
                f'BC {loss_bc.item():.4e} | '
                f'PDE {loss_f.item():.4e} | '
                f'CONS {loss_cons.item():.4e}'
            )

    elapsed = time.time() - start
    print(f'[{model_name}] Training done in {elapsed:.2f} s')
    return history, elapsed

## 11. 开始训练

建议课堂上先跑普通 PINN，再跑 Conservation-PINN，这样学生更容易理解“只多加了一个守恒项”。

In [ ]:
# baseline_history, baseline_time = train_model(
#     baseline_model,
#     model_name='Baseline PINN',
#     use_conservation=False,
# )

# cons_history, cons_time = train_model(
#     cons_model,
#     model_name='Conservation-PINN',
#     use_conservation=True,
# )

## 12. 训练曲线可视化

完成训练后，取消注释并运行下面的单元。

In [ ]:
# plt.figure(figsize=(7, 4.5))
# plt.semilogy(baseline_history['total'], label='Baseline PINN')
# plt.semilogy(cons_history['total'], label='Conservation-PINN')
# plt.xlabel('Epoch')
# plt.ylabel('Total loss')
# plt.title('总损失对比')
# plt.grid(alpha=0.3)
# plt.legend()
# plt.show()
#
# fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
# for ax, key, title in zip(
#     axes,
#     ['ic', 'bc', 'f', 'cons'],
#     ['IC loss', 'BC loss', 'PDE loss', 'Conservation loss']
# ):
#     ax.semilogy(baseline_history[key], label='Baseline PINN')
#     ax.semilogy(cons_history[key], label='Conservation-PINN')
#     ax.set_title(title)
#     ax.set_xlabel('Epoch')
#     ax.grid(alpha=0.3)
# axes[0].legend()
# plt.tight_layout()
# plt.show()

## 13. 网格评估函数

我们后面要比较：

1. $L_2$ 相对误差；
2. 守恒量误差；
3. 热力图与时间切片图。

In [ ]:
def evaluate_model(model, c, nx=200, nt=200):
    x_eval = np.linspace(0.0, 1.0, nx)
    t_eval = np.linspace(0.0, 1.0, nt)
    X, T = np.meshgrid(x_eval, t_eval)

    XT = np.hstack([X.reshape(-1, 1), T.reshape(-1, 1)])
    xt_tensor = torch.tensor(XT, dtype=torch.float32, device=device)

    model.eval()
    with torch.no_grad():
        u_pred = model(xt_tensor[:, :1], xt_tensor[:, 1:2]).cpu().numpy().reshape(nt, nx)
    model.train()

    u_exact = exact_solution(X, T, c)
    abs_err = np.abs(u_pred - u_exact)
    l2_rel = np.linalg.norm(u_pred - u_exact) / np.linalg.norm(u_exact)

    pred_mass = np.mean(u_pred, axis=1)
    exact_mass = np.mean(u_exact, axis=1)
    cons_err = np.abs(pred_mass - M0_exact)

    return {
        'x_eval': x_eval,
        't_eval': t_eval,
        'u_pred': u_pred,
        'u_exact': u_exact,
        'abs_err': abs_err,
        'l2_rel': l2_rel,
        'pred_mass': pred_mass,
        'exact_mass': exact_mass,
        'cons_err': cons_err,
    }

## 14. 运行评估

In [ ]:
# baseline_eval = evaluate_model(baseline_model, c)
# cons_eval = evaluate_model(cons_model, c)
#
# print(f"Baseline PINN L2 relative error: {baseline_eval['l2_rel']:.4e}")
# print(f"Conservation-PINN L2 relative error: {cons_eval['l2_rel']:.4e}")
# print(f"Baseline mean conservation error: {np.mean(baseline_eval['cons_err']):.4e}")
# print(f"Conservation-PINN mean conservation error: {np.mean(cons_eval['cons_err']):.4e}")

## 15. 守恒量误差曲线

守恒量误差定义为：

$$E_{cons}(t)=|M_\theta(t)-M_0|$$

In [ ]:
# plt.figure(figsize=(7, 4.5))
# plt.plot(baseline_eval['t_eval'], baseline_eval['cons_err'], label='Baseline PINN')
# plt.plot(cons_eval['t_eval'], cons_eval['cons_err'], label='Conservation-PINN')
# plt.xlabel('t')
# plt.ylabel('Conservation error')
# plt.title('守恒量误差对比')
# plt.grid(alpha=0.3)
# plt.legend()
# plt.show()

## 16. 预测解 / 真解 / 绝对误差

In [ ]:
# fig, axes = plt.subplots(2, 3, figsize=(14, 7.2))
#
# im0 = axes[0, 0].imshow(baseline_eval['u_pred'], extent=[0,1,0,1], origin='lower', aspect='auto', cmap='viridis')
# axes[0, 0].set_title('Baseline PINN 预测解')
# plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)
#
# im1 = axes[0, 1].imshow(baseline_eval['u_exact'], extent=[0,1,0,1], origin='lower', aspect='auto', cmap='viridis')
# axes[0, 1].set_title('解析真解')
# plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)
#
# im2 = axes[0, 2].imshow(baseline_eval['abs_err'], extent=[0,1,0,1], origin='lower', aspect='auto', cmap='magma')
# axes[0, 2].set_title('Baseline PINN 绝对误差')
# plt.colorbar(im2, ax=axes[0, 2], fraction=0.046)
#
# im3 = axes[1, 0].imshow(cons_eval['u_pred'], extent=[0,1,0,1], origin='lower', aspect='auto', cmap='viridis')
# axes[1, 0].set_title('Conservation-PINN 预测解')
# plt.colorbar(im3, ax=axes[1, 0], fraction=0.046)
#
# im4 = axes[1, 1].imshow(cons_eval['u_exact'], extent=[0,1,0,1], origin='lower', aspect='auto', cmap='viridis')
# axes[1, 1].set_title('解析真解')
# plt.colorbar(im4, ax=axes[1, 1], fraction=0.046)
#
# im5 = axes[1, 2].imshow(cons_eval['abs_err'], extent=[0,1,0,1], origin='lower', aspect='auto', cmap='magma')
# axes[1, 2].set_title('Conservation-PINN 绝对误差')
# plt.colorbar(im5, ax=axes[1, 2], fraction=0.046)
#
# for ax in axes.ravel():
#     ax.set_xlabel('x')
#     ax.set_ylabel('t')
#
# plt.tight_layout()
# plt.show()

## 17. 时间切片对比图

In [ ]:
# time_slices = [0.0, 0.25, 0.5, 0.75, 1.0]
#
# fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
# for ax, title, result in [
#     (axes[0], 'Baseline PINN vs 解析解', baseline_eval),
#     (axes[1], 'Conservation-PINN vs 解析解', cons_eval),
# ]:
#     for ts in time_slices:
#         idx = np.argmin(np.abs(result['t_eval'] - ts))
#         ax.plot(result['x_eval'], result['u_exact'][idx], 'k--', linewidth=1.0)
#         ax.plot(result['x_eval'], result['u_pred'][idx], linewidth=1.5, label=f't={result["t_eval"][idx]:.2f}')
#     ax.set_xlabel('x')
#     ax.set_title(title)
#     ax.grid(alpha=0.3)
#
# axes[0].set_ylabel('u(x,t)')
# axes[1].legend()
# plt.tight_layout()
# plt.show()

## 18. 结果汇总表

In [ ]:
# summary = {
#     'Baseline PINN': {
#         'final_total_loss': baseline_history['total'][-1],
#         'final_pde_loss': baseline_history['f'][-1],
#         'final_cons_loss': baseline_history['cons'][-1],
#         'l2_relative_error': baseline_eval['l2_rel'],
#         'mean_conservation_error': np.mean(baseline_eval['cons_err']),
#         'train_time_s': baseline_time,
#     },
#     'Conservation-PINN': {
#         'final_total_loss': cons_history['total'][-1],
#         'final_pde_loss': cons_history['f'][-1],
#         'final_cons_loss': cons_history['cons'][-1],
#         'l2_relative_error': cons_eval['l2_rel'],
#         'mean_conservation_error': np.mean(cons_eval['cons_err']),
#         'train_time_s': cons_time,
#     },
# }
#
# for name, info in summary.items():
#     print(name)
#     for key, value in info.items():
#         print(f'  {key:24s}: {value:.4e}' if isinstance(value, float) else f'  {key}: {value}')
#     print()

## 19. 课堂讨论与后续实验建议

1. 当 $\lambda_{cons}$ 增大时，守恒量误差是否明显下降？
2. 守恒量保持更好时，$L_2$ 误差一定同步下降吗？
3. 周期边界的一阶导一致性是否重要？
4. 后续是否可以把这里的 `MLP` 换成 `Res-PINN`，做成 `Conservation-ResPINN`？
5. 是否可以把这个思路推广到 Burgers 方程或更一般的守恒律问题？